# 🎛️ Módulo 7: Operaciones con Señales
### *El ADN de las Señales: Transformar, Combinar y Dominar*

---

> **¿Sabías que** procesar audio, video o señales médicas es, en esencia, manipular el tiempo y la amplitud? Este notebook te llevará desde los fundamentos hasta aplicaciones como radares Doppler y filtros de audio.

**Al finalizar este módulo serás capaz de:**
- Aplicar desplazamiento, escalamiento e inversión sobre señales continuas $x(t)$ y discretas $x[n]$
- Combinar señales mediante suma y multiplicación punto a punto
- Reconocer los efectos de diezmado e interpolación en señales discretas
- Resolver problemas compuestos de transformaciones en el orden correcto

---

In [ ]:
# ============================================================
# Celda de configuración — ejecutar primero
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Estilo visual uniforme para todo el notebook
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#ccccff',
    'xtick.color':      '#ccccff',
    'ytick.color':      '#ccccff',
    'text.color':       '#eeeeee',
    'grid.color':       '#333355',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'axes.titlecolor':  '#00ffcc',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'font.family':      'monospace',
})

COLOR_ORIG  = '#00e5ff'   # señal original
COLOR_TRANS = '#ff6b6b'   # señal transformada
COLOR_EXTRA = '#ffd93d'   # señal auxiliar / resultado
COLOR_WARN  = '#ff9500'   # advertencias

print('✅ Entorno listo. ¡Comencemos!')

---
## 📺 Sección 1 — El ADN de las Señales

### Concepto central: ¿Qué es una transformación de señal?

Toda señal $x(t)$ (o $x[n]$) lleva codificada información en **dos dimensiones**:

| Dimensión | Parámetro | Efecto al modificarlo |
|-----------|-----------|----------------------|
| Tiempo (eje horizontal) | $t$ o $n$ | Desplazamiento, compresión, inversión |
| Amplitud (eje vertical) | $A$ | Ganancia, atenuación |

Manipular estas dimensiones produce señales nuevas que modelan ecos, radar, cambios de velocidad de audio y más.

### Forma general de una señal transformada

$$y(t) = A \cdot x(at - t_0)$$

donde:
- $A$ → **amplitud** (ganancia si $|A|>1$, atenuación si $|A|<1$; inversión de amplitud si $A<0$)
- $a$ → **escala temporal** (compresión si $|a|>1$, expansión si $|a|<1$; inversión temporal si $a<0$)
- $t_0$ → **desplazamiento** (retardo si $t_0>0$, adelanto si $t_0<0$)

> 🔑 **Regla de oro:** El orden correcto de aplicación es:  
> $1.$ Desplazamiento $\;\rightarrow\;$ $2.$ Inversión / Escalamiento temporal $\;\rightarrow\;$ $3.$ Escalamiento de amplitud

---
## 📋 Sección 2 — Las Reglas del Juego
### Tabla comparativa: Tiempo Continuo (TC) vs. Tiempo Discreto (TD)

| Operación | Tiempo Continuo $x(t)$ | Tiempo Discreto $x[n]$ | Efecto visual |
|-----------|----------------------|----------------------|---------------|
| **Desplazamiento** | $x(t - t_0)$ | $x[n - n_0]$ | Derecha ($>0$) / Izquierda ($<0$) |
| **Escalamiento** | $x(at)$ | $x[an]$ | Compresión ($|a|>1$) / Expansión ($|a|<1$) |
| **Inversión** | $x(-t)$ | $x[-n]$ | Espejo respecto al eje vertical |
| **Suma** | $x_1(t)+x_2(t)$ | $x_1[n]+x_2[n]$ | Punto a punto / muestra a muestra |
| **Multiplicación** | $x_1(t)\cdot x_2(t)$ | $x_1[n]\cdot x_2[n]$ | Punto a punto / muestra a muestra |

---
### ⚠️ Alerta de Ingeniería — Mundo Discreto

En tiempo discreto, el índice $n$ **debe ser entero**. Por eso el escalamiento $x[an]$ tiene consecuencias únicas:

- Si $a > 1$ → **Diezmado**: se descartan muestras intermedias (pérdida de información).
- Si $a < 1$ → **Interpolación**: aparecen "huecos" entre muestras que requieren ser estimados.

Esto **no ocurre** en tiempo continuo, donde $t$ puede tomar cualquier valor real.

---
## 🔬 Sección 3 — Laboratorio Virtual de Transformaciones

### 3.1 Desplazamiento temporal

In [ ]:
# ----------------------------------------------------------------
# DESPLAZAMIENTO: x(t - t0)  y  x[n - n0]
# ----------------------------------------------------------------
def pulso_triangular(t, centro=0, ancho=2):
    """Pulso triangular centrado en `centro` con ancho total `ancho`."""
    return np.maximum(0, 1 - np.abs((t - centro) / (ancho / 2)))

t  = np.linspace(-6, 10, 1000)
t0 = 3          # retardo de 3 unidades

x_tc      = pulso_triangular(t)
x_tc_desp = pulso_triangular(t - t0)   # x(t - 3)

# --- Tiempo discreto ---
n       = np.arange(-6, 11)
n0      = 3
x_td      = pulso_triangular(n.astype(float))
x_td_desp = pulso_triangular((n - n0).astype(float))  # x[n - 3]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle('Desplazamiento Temporal', fontsize=15, color='#00ffcc', fontweight='bold')

# TC original
axes[0,0].plot(t, x_tc, color=COLOR_ORIG, lw=2)
axes[0,0].set_title('TC original: $x(t)$');  axes[0,0].grid(True)
axes[0,0].axvline(0, color='white', lw=0.5, ls=':')

# TC desplazada
axes[0,1].plot(t, x_tc_desp, color=COLOR_TRANS, lw=2)
axes[0,1].set_title(f'TC desplazada: $x(t - {t0})$  →  retardo');  axes[0,1].grid(True)
axes[0,1].axvline(0, color='white', lw=0.5, ls=':')
axes[0,1].plot(t, x_tc, color=COLOR_ORIG, lw=1, alpha=0.3, ls='--')  # referencia

# TD original
axes[1,0].stem(n, x_td, linefmt=COLOR_ORIG, markerfmt=f'o', basefmt='gray')
axes[1,0].set_title('TD original: $x[n]$');  axes[1,0].grid(True)

# TD desplazada
axes[1,1].stem(n, x_td_desp, linefmt=COLOR_TRANS, markerfmt=f'o', basefmt='gray')
axes[1,1].set_title(f'TD desplazada: $x[n - {n0}]$  →  retardo');  axes[1,1].grid(True)

for ax in axes.flat:
    ax.set_xlabel('Tiempo'); ax.set_ylim(-0.15, 1.3)

plt.tight_layout()
plt.show()
print(f'📌 La señal se desplazó {t0} unidades a la DERECHA (retardo).')

### 3.2 Escalamiento temporal (compresión y expansión)

Para la señal continua $y(t) = x(at)$:
$$a > 1 \Rightarrow \text{Compresión en tiempo} \qquad a < 1 \Rightarrow \text{Expansión en tiempo}$$

En tiempo discreto la historia es diferente — veremos el efecto de **diezmado**.

In [ ]:
# ----------------------------------------------------------------
# ESCALAMIENTO TEMPORAL
# ----------------------------------------------------------------
t = np.linspace(-5, 5, 1000)

x_orig       = pulso_triangular(t)
x_comprimida = pulso_triangular(2 * t)     # a=2: compresión
x_expandida  = pulso_triangular(0.5 * t)   # a=0.5: expansión

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Escalamiento Temporal en Tiempo Continuo', fontsize=14, color='#00ffcc', fontweight='bold')

datos = [
    (x_orig,       COLOR_ORIG,  '$x(t)$ — original',         ''),
    (x_comprimida, COLOR_TRANS, '$x(2t)$ — compresión $a=2$', '← más estrecha'),
    (x_expandida,  COLOR_EXTRA, '$x(0.5t)$ — expansión $a=0.5$', '← más ancha'),
]

for ax, (y, col, titulo, nota) in zip(axes, datos):
    ax.plot(t, y, color=col, lw=2)
    ax.plot(t, x_orig, color=COLOR_ORIG, lw=1, alpha=0.25, ls='--')  # referencia
    ax.set_title(f'{titulo}\n{nota}', fontsize=10)
    ax.set_xlabel('$t$'); ax.grid(True); ax.set_ylim(-0.15, 1.3)
    ax.axvline(0, color='white', lw=0.5, ls=':')

plt.tight_layout()
plt.show()

### 3.3 Diezmado e Interpolación en Tiempo Discreto

En TD, el escalamiento $x[an]$ fuerza $an$ a ser entero. Esto genera dos fenómenos:

$$x[2n] \Rightarrow \text{solo se conservan muestras pares}\quad (\text{diezmado por 2})$$
$$x[n/2] \Rightarrow \text{se necesitan muestras intermedias}\quad (\text{interpolación})$$

In [ ]:
# ----------------------------------------------------------------
# DIEZMADO E INTERPOLACIÓN EN TIEMPO DISCRETO
# ----------------------------------------------------------------
n_orig = np.arange(-5, 11)
x_orig_td = pulso_triangular(n_orig.astype(float))

# --- Diezmado: x[2n] — tomamos una de cada dos muestras ---
n_diezm = np.arange(-2, 6)           # índices después de a=2
x_diezm  = pulso_triangular((2 * n_diezm).astype(float))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Escalamiento en Tiempo Discreto: Diezmado vs. Interpolación',
             fontsize=13, color='#00ffcc', fontweight='bold')

# Original
markerline, stemlines, baseline = axes[0].stem(n_orig, x_orig_td, basefmt='gray')
plt.setp(stemlines, color=COLOR_ORIG); plt.setp(markerline, color=COLOR_ORIG, markersize=7)
axes[0].set_title('$x[n]$ — original (16 muestras)'); axes[0].grid(True)

# Diezmado
ml, sl, bl = axes[1].stem(n_diezm, x_diezm, basefmt='gray')
plt.setp(sl, color=COLOR_TRANS); plt.setp(ml, color=COLOR_TRANS, markersize=7)

# Marcar muestras "eliminadas" en gris
n_eliminadas = np.array([n for n in n_orig if n not in n_diezm * 1])
# Todas las impares que sí existían
mask_impar = (n_orig % 2 != 0)
axes[1].scatter(n_orig[mask_impar], x_orig_td[mask_impar],
                marker='x', color=COLOR_WARN, s=80, zorder=5, label='Muestras perdidas')
axes[1].set_title('$x[2n]$ — diezmado por 2\n⚠️ Muestras impares perdidas (×)')
axes[1].grid(True); axes[1].legend(fontsize=8)

# Interpolación (x[n/2] expandido con ceros)
n_interp = np.arange(-10, 21)
x_interp  = np.zeros(len(n_interp))
for i, ni in enumerate(n_interp):
    if ni % 2 == 0:
        idx = np.where(n_orig == ni // 2)[0]
        if len(idx): x_interp[i] = x_orig_td[idx[0]]

# Muestras interpoladas (huecos)
mask_huecos = (n_interp % 2 != 0)
ml2, sl2, bl2 = axes[2].stem(n_interp, x_interp, basefmt='gray')
plt.setp(sl2, color=COLOR_EXTRA); plt.setp(ml2, color=COLOR_EXTRA, markersize=7)
axes[2].scatter(n_interp[mask_huecos], np.zeros(mask_huecos.sum()),
                marker='?', color='gray', s=60, zorder=5, alpha=0.5)
axes[2].scatter(n_interp[mask_huecos],
                [0.05]*mask_huecos.sum(),
                marker='v', color='gray', s=50, zorder=5, alpha=0.6, label='? requiere interpolación')
axes[2].set_title('$x[n/2]$ — expansión por 2\n? Huecos requieren interpolación')
axes[2].grid(True); axes[2].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel('$n$'); ax.set_ylim(-0.2, 1.4)

plt.tight_layout()
plt.show()

print('📌 En diezmado se pierden N/2 muestras. En interpolación se necesita un algoritmo')
print('   (ej. lineal, sinc) para estimar los valores en los índices fraccionarios.')

### 3.4 Inversión temporal

La inversión refleja la señal respecto al eje vertical:
$$y(t) = x(-t) \qquad y[n] = x[-n]$$

Es equivalente a escalar el tiempo con $a = -1$.

In [ ]:
# ----------------------------------------------------------------
# INVERSIÓN TEMPORAL
# ----------------------------------------------------------------
def x_asimetrica(t):
    """Señal asimétrica para apreciar la inversión."""
    return (pulso_triangular(t, centro=1, ancho=2) +
            0.5 * pulso_triangular(t, centro=-2, ancho=1))

t = np.linspace(-6, 6, 1000)
n = np.arange(-6, 7)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
fig.suptitle('Inversión Temporal $x(-t)$ / $x[-n]$', fontsize=14,
             color='#00ffcc', fontweight='bold')

# TC
axes[0,0].plot(t, x_asimetrica(t),  color=COLOR_ORIG,  lw=2)
axes[0,0].set_title('TC: $x(t)$'); axes[0,0].grid(True)

axes[0,1].plot(t, x_asimetrica(-t), color=COLOR_TRANS, lw=2)
axes[0,1].plot(t, x_asimetrica(t),  color=COLOR_ORIG,  lw=1, alpha=0.25, ls='--')
axes[0,1].set_title('TC: $x(-t)$ — espejo horizontal'); axes[0,1].grid(True)
axes[0,1].axvline(0, color='white', lw=1, ls=':')

# TD
ml, sl, bl = axes[1,0].stem(n, x_asimetrica(n.astype(float)), basefmt='gray')
plt.setp(sl, color=COLOR_ORIG); plt.setp(ml, color=COLOR_ORIG)
axes[1,0].set_title('TD: $x[n]$'); axes[1,0].grid(True)

ml2, sl2, bl2 = axes[1,1].stem(n, x_asimetrica((-n).astype(float)), basefmt='gray')
plt.setp(sl2, color=COLOR_TRANS); plt.setp(ml2, color=COLOR_TRANS)
axes[1,1].set_title('TD: $x[-n]$ — espejo horizontal'); axes[1,1].grid(True)
axes[1,1].axvline(0, color='white', lw=1, ls=':')

for ax in axes.flat:
    ax.set_xlabel('tiempo'); ax.set_ylim(-0.1, 1.5)

plt.tight_layout()
plt.show()

### 3.5 Aritmética de Señales: Suma y Multiplicación

Dadas dos señales $x_1[n]$ y $x_2[n]$, las operaciones aritméticas se realizan **muestra a muestra**:

$$z[n] = x_1[n] + x_2[n] \qquad \text{(suma)}$$
$$z[n] = x_1[n] \cdot x_2[n] \qquad \text{(producto)}$$

**Concepto clave: Soporte de la señal**  
El soporte es el conjunto de índices donde $x[n] \neq 0$. En la multiplicación, el soporte del resultado es la **intersección** de los soportes de ambas señales.

In [ ]:
# ----------------------------------------------------------------
# ARITMÉTICA DE SEÑALES
# ----------------------------------------------------------------
# Escalón unitario discreto
def u(n): return (n >= 0).astype(float)

n = np.arange(-3, 10)

x1 = u(n)        # escalón unitario
x2 = u(n - 4)    # escalón retardado 4 muestras

suma  = x1 + x2
resta = x1 - x2         # pulso cuadrado de ancho 4
prod  = x1 * x2

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Aritmética de Señales Discretas', fontsize=14, color='#00ffcc', fontweight='bold')

def plot_stem(ax, n, x, color, titulo):
    ml, sl, bl = ax.stem(n, x, basefmt='gray')
    plt.setp(sl, color=color); plt.setp(ml, color=color, markersize=7)
    ax.set_title(titulo); ax.grid(True); ax.set_xlabel('$n$')

plot_stem(axes[0,0], n, x1,    COLOR_ORIG,  '$x_1[n] = u[n]$')
plot_stem(axes[0,1], n, x2,    COLOR_EXTRA, '$x_2[n] = u[n-4]$')
plot_stem(axes[0,2], n, suma,  COLOR_TRANS, '$x_1[n] + x_2[n]$ — suma')
plot_stem(axes[1,0], n, resta, '#a0ff70',   '$x_1[n] - x_2[n]$ — pulso cuadrado de ancho 4')
plot_stem(axes[1,1], n, prod,  COLOR_WARN,  '$x_1[n] \\cdot x_2[n]$ — soporte: intersección')

# Resaltar el soporte en el producto
mask_soporte = (prod != 0)
axes[1,1].scatter(n[mask_soporte], prod[mask_soporte],
                  s=120, color='white', zorder=5, alpha=0.4)

axes[1,2].axis('off')
nota = (
    "Observaciones:\n\n"
    "• u[n] - u[n-4] crea un\n  pulso cuadrado de ancho 4.\n\n"
    "• El producto tiene soporte\n  solo donde AMBAS señales\n  son distintas de cero.\n\n"
    "• Suma de escalones crea\n  una rampa escalonada."
)
axes[1,2].text(0.05, 0.5, nota, transform=axes[1,2].transAxes,
               fontsize=10, color='#ccccff', va='center',
               bbox=dict(facecolor='#2a2a4a', edgecolor='#00ffcc', boxstyle='round,pad=0.5'))

plt.tight_layout()
plt.show()

---
## 📖 Sección 4 — Guía de Resolución Paso a Paso

### Ejemplo: $y[n] = x_1[n-1] + 2x_2[n]$

Dadas las secuencias:
$$x_1[n] = \{\underset{\uparrow}{1}, 2, 3\} \qquad x_2[n] = \{\underset{\uparrow}{1}, 1, 1\}$$

**Pasos:**
1. **Desplazar** $x_1[n]$ → $x_1[n-1]$
2. **Escalar amplitud** $x_2[n]$ → $2x_2[n]$
3. **Sumar** punto a punto

In [ ]:
# ----------------------------------------------------------------
# EJEMPLO PASO A PASO: y[n] = x1[n-1] + 2*x2[n]
# ----------------------------------------------------------------
# Definimos las secuencias con su índice
n_x1 = np.array([0, 1, 2])        # x1[n] definida para n=0,1,2
vals_x1 = np.array([1, 2, 3])

n_x2 = np.array([0, 1, 2])        # x2[n] definida para n=0,1,2
vals_x2 = np.array([1, 1, 1])

# Paso 1: Desplazamiento x1[n-1] -> índices se mueven +1
n_x1_desp  = n_x1 + 1
vals_x1_d  = vals_x1.copy()

# Paso 2: Escalamiento de amplitud 2*x2[n]
vals_x2_esc = 2 * vals_x2

# Paso 3: Suma en el dominio común n = [0,1,2,3]
n_total = np.arange(0, 4)
y = np.zeros(len(n_total))

for i, ni in enumerate(n_total):
    if ni in n_x1_desp:
        y[i] += vals_x1_d[np.where(n_x1_desp == ni)[0][0]]
    if ni in n_x2:
        y[i] += vals_x2_esc[np.where(n_x2 == ni)[0][0]]

# ---- Visualización con 4 paneles ----
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Solución Paso a Paso: $y[n] = x_1[n-1] + 2x_2[n]$',
             fontsize=13, color='#00ffcc', fontweight='bold')

def stem_c(ax, n, v, col, title, highlight=None):
    ml, sl, bl = ax.stem(n, v, basefmt='gray')
    plt.setp(sl, color=col, lw=2); plt.setp(ml, color=col, markersize=9)
    if highlight is not None:
        ax.scatter(n[highlight], v[highlight], s=150, color='white', zorder=6, alpha=0.6)
    ax.set_title(title, fontsize=9.5)
    ax.grid(True); ax.set_xlabel('$n$')
    ax.set_ylim(-0.5, 7)
    # Anotar valores
    for ni, vi in zip(n, v):
        if vi != 0:
            ax.text(ni, vi + 0.2, str(int(vi)), ha='center',
                    color='white', fontsize=9)

stem_c(axes[0], n_x1,      vals_x1,     COLOR_ORIG,  'Paso 0: $x_1[n]$')
stem_c(axes[1], n_x1_desp, vals_x1_d,   COLOR_TRANS, 'Paso 1: $x_1[n-1]$\n(desplazado +1)')
stem_c(axes[2], n_x2,      vals_x2_esc, COLOR_EXTRA, 'Paso 2: $2x_2[n]$\n(amplitud ×2)')
stem_c(axes[3], n_total,   y,           '#ff9500',   'Paso 3: $y[n] = $ suma\n$y[n]=\{2,3,4,3\}$',
       highlight=np.ones(len(n_total), dtype=bool))

plt.tight_layout()
plt.show()

print('Secuencia resultante:')
for ni, yi in zip(n_total, y):
    marca = ' ←' if ni == 0 else ''
    print(f'  y[{ni}] = {int(yi)}{marca}')

---
## 🎮 Sección 5 — Retos Interactivos

### 🎯 Reto 1: Filtro de Diezmado

**Misión:** Tienes una señal de temperatura $x[n]$ con 10 muestras. Aplica diezmado por 2 ($a=2$). ¿Cuántas muestras quedan?

In [ ]:
# ----------------------------------------------------------------
# RETO 1: Diezmado
# ----------------------------------------------------------------
# Señal de temperatura simulada
n_sensor = np.arange(0, 10)
temp = np.array([20.1, 20.5, 21.0, 21.8, 22.3, 22.0, 21.5, 21.0, 20.7, 20.4])

# Diezmado por 2: tomamos índices pares
n_diezm = n_sensor[::2]
temp_d  = temp[::2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('🎯 Reto 1 — Sensor de Temperatura con Diezmado', fontsize=13,
             color='#00ffcc', fontweight='bold')

ml, sl, bl = ax1.stem(n_sensor, temp, basefmt='gray')
plt.setp(sl, color=COLOR_ORIG); plt.setp(ml, color=COLOR_ORIG, markersize=8)
ax1.set_title(f'Original: {len(n_sensor)} muestras'); ax1.set_xlabel('muestra $n$')
ax1.set_ylabel('Temperatura (°C)'); ax1.grid(True)

ml2, sl2, bl2 = ax2.stem(n_diezm, temp_d, basefmt='gray')
plt.setp(sl2, color=COLOR_TRANS); plt.setp(ml2, color=COLOR_TRANS, markersize=8)
# Muestras eliminadas
mask_elim = np.isin(n_sensor, n_diezm, invert=True)
ax2.scatter(n_sensor[mask_elim], temp[mask_elim],
            marker='x', s=100, color=COLOR_WARN, zorder=5, label='Muestras perdidas')
ax2.set_title(f'Diezmado (a=2): {len(n_diezm)} muestras'); ax2.set_xlabel('muestra $n$')
ax2.set_ylabel('Temperatura (°C)'); ax2.grid(True); ax2.legend()

plt.tight_layout()
plt.show()

print(f'✅ Respuesta: de {len(n_sensor)} muestras quedan {len(n_diezm)} muestras.')
print(f'   Almacenamiento reducido al {100*len(n_diezm)//len(n_sensor)}%.')

### 🎯 Reto 2: Construcción de Pulsos con Escalones

**Misión:** Construye un pulso cuadrado de ancho 4 usando **solo escalones unitarios**.

$$\text{Pulso}[n] = u[n] - u[n-4]$$

In [ ]:
# ----------------------------------------------------------------
# RETO 2: Pulso cuadrado con escalones
# ----------------------------------------------------------------
n = np.arange(-3, 10)

u_n   = u(n)
u_n4  = u(n - 4)
pulso = u_n - u_n4

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('🎯 Reto 2 — Construcción de Pulso Cuadrado', fontsize=13,
             color='#00ffcc', fontweight='bold')

plot_stem(axes[0], n, u_n,  COLOR_ORIG,  '$x_1[n] = u[n]$')
plot_stem(axes[1], n, u_n4, COLOR_EXTRA, '$x_2[n] = u[n-4]$')
plot_stem(axes[2], n, pulso, COLOR_TRANS, '$y[n] = u[n] - u[n-4]$\nPulso cuadrado de ancho 4')

# Sombrear el pulso
n_pulso = n[pulso == 1]
for ni in n_pulso:
    axes[2].axvspan(ni-0.4, ni+0.4, alpha=0.15, color=COLOR_TRANS)

for ax in axes:
    ax.set_ylim(-0.3, 1.5)

plt.tight_layout()
plt.show()

print('✅ El pulso cuadrado tiene exactamente', int(pulso.sum()), 'muestras distintas de cero.')

### 🎯 Reto 3: El Radar Doppler (TC + TD)

**Misión:** Sincronizar una señal emitida con una recibida.

Una señal de radar emitida llega con:
- **Atenuación** por distancia: $A = 0.8$
- **Compresión** por efecto Doppler (objeto acercándose): $a = 1.2$  
- **Retardo** de propagación: $t_0 = 4$

$$y(t) = 0.8 \cdot x(1.2t - 4)$$

Primero factorizamos para entender el desplazamiento:
$$y(t) = 0.8 \cdot x\!\left(1.2\left(t - \tfrac{4}{1.2}\right)\right) = 0.8 \cdot x\!\left(1.2\left(t - 3.33\right)\right)$$

In [ ]:
# ----------------------------------------------------------------
# RETO 3: RADAR DOPPLER
# ----------------------------------------------------------------
def x_radar(t):
    """Pulso de radar: suma de dos triangulares."""
    return (pulso_triangular(t, centro=0, ancho=3) +
            0.4 * pulso_triangular(t, centro=3, ancho=1.5))

t = np.linspace(-2, 12, 2000)

A   = 0.8    # atenuación
a   = 1.2    # compresión Doppler
t0  = 4      # retardo total en la ecuación original

x_emit = x_radar(t)
y_recv = A * x_radar(a * t - t0)      # señal recibida

# Digitalización (muestreo)
Ts = 0.3   # período de muestreo
n_dig   = np.arange(0, int(12/Ts))
t_dig   = n_dig * Ts
y_dig   = A * x_radar(a * t_dig - t0)

fig = plt.figure(figsize=(14, 8))
gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle('🎯 Reto 3 — Radar Doppler: $y(t) = 0.8\\,x(1.2t - 4)$',
             fontsize=14, color='#00ffcc', fontweight='bold')

# Panel 1: señales superpuestas
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(t, x_emit, color=COLOR_ORIG,  lw=2.5, label='$x(t)$ — emitida')
ax1.plot(t, y_recv, color=COLOR_TRANS, lw=2.5, label='$y(t) = 0.8x(1.2t-4)$ — recibida')
ax1.fill_between(t, x_emit, y_recv, alpha=0.1, color='white')
ax1.set_title('Señal emitida vs. recibida en TC'); ax1.grid(True)
ax1.set_xlabel('$t$ (s)'); ax1.set_ylabel('Amplitud')
ax1.legend(fontsize=10)

# Anotaciones
ax1.annotate('Retardo\n$t_0/a = 3.33$ s',
             xy=(3.33, 0.05), xytext=(1.5, -0.25),
             arrowprops=dict(arrowstyle='->', color='white'), color='white', fontsize=9)
ax1.annotate('Compresión\n$a=1.2$',
             xy=(4.5, 0.6), xytext=(6.5, 0.7),
             arrowprops=dict(arrowstyle='->', color=COLOR_TRANS), color=COLOR_TRANS, fontsize=9)

# Panel 2: análisis de parámetros
ax2 = fig.add_subplot(gs[1, 0])
ax2.axis('off')
tabla = (
    "Análisis de parámetros:\n\n"
    "  Amplitud A = 0.8  → atenuación (< 1)\n"
    "  Escala  a = 1.2  → compresión (> 1)\n"
    "                     objeto acercándose\n"
    "  Delay  t₀ = 4   → retardo de propagación\n\n"
    "  Desplazamiento real: t₀/a = 4/1.2 ≈ 3.33 s"
)
ax2.text(0.05, 0.5, tabla, transform=ax2.transAxes,
         fontsize=10, color='#ccccff', va='center', family='monospace',
         bbox=dict(facecolor='#2a2a4a', edgecolor='#00ffcc', boxstyle='round,pad=0.6'))

# Panel 3: digitalización
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(t, y_recv, color=COLOR_TRANS, lw=1.5, alpha=0.4, label='Señal analógica')
ml, sl, bl = ax3.stem(t_dig, y_dig, basefmt='gray')
plt.setp(sl, color=COLOR_EXTRA); plt.setp(ml, color=COLOR_EXTRA, markersize=6)
ax3.set_title(f'Digitalización ($T_s = {Ts}$ s) → $y[n]$')
ax3.set_xlabel('$t$ (s)'); ax3.grid(True)
ax3.legend(fontsize=9)

plt.show()

print('✅ Parámetros identificados:')
print(f'   Potencia relativa = {A} (perdió {(1-A)*100:.0f}% de potencia)')
print(f'   Factor de compresión = {a} (objeto acercándose)')
print(f'   Retardo efectivo = {t0/a:.2f} s')

---
## 📊 Sección 6 — Banco de Ejercicios y Evaluación

### Nivel Pro: Rampa Unitaria con Transformaciones

**Problema:** Graficar $y(t) = 2r(-t + 2)$

**Estrategia:** Factorizar para identificar parámetros:
$$y(t) = 2r(-(t - 2))$$

Entonces:
1. $r(t)$ → rampa unitaria
2. $r(t - 2)$ → desplazamiento +2 unidades a la derecha
3. $r(-(t-2))$ → inversión temporal respecto a $t = 2$
4. $2r(-(t-2))$ → amplitud × 2

In [ ]:
# ----------------------------------------------------------------
# NIVEL PRO: y(t) = 2*r(-t + 2)
# ----------------------------------------------------------------
def rampa(t): return np.maximum(0, t)

t = np.linspace(-2, 6, 1000)

paso0 = rampa(t)            # r(t)
paso1 = rampa(t - 2)        # r(t-2): desplazar
paso2 = rampa(-(t - 2))     # r(-(t-2)): invertir
paso3 = 2 * rampa(-(t - 2)) # 2*r(-(t-2)): amplificar

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Nivel Pro: $y(t) = 2r(-t+2) = 2r(-(t-2))$ — Pasos de construcción',
             fontsize=12, color='#00ffcc', fontweight='bold')

pasos = [
    (paso0, COLOR_ORIG,  'Paso 1: $r(t)$ — rampa original'),
    (paso1, COLOR_EXTRA, 'Paso 2: $r(t-2)$ — desplazar +2'),
    (paso2, COLOR_TRANS, 'Paso 3: $r(-(t-2))$ — invertir en $t=2$'),
    (paso3, COLOR_WARN,  'Paso 4: $2r(-(t-2))$ — amplitud ×2  ✅'),
]

for ax, (y, col, titulo) in zip(axes.flat, pasos):
    ax.plot(t, y, color=col, lw=2.5)
    ax.axvline(0, color='white', lw=0.5, ls=':')
    ax.axvline(2, color='white', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(titulo); ax.grid(True)
    ax.set_xlabel('$t$'); ax.set_ylabel('Amplitud')
    ax.set_ylim(-0.3, 5)
    ax.text(2.05, 0.3, '$t=2$', color='white', fontsize=8)

plt.tight_layout()
plt.show()

print('✅ Clave: factorizar como r(-(t-2)) deja ver el desplazamiento ANTES de la inversión.')
print('   Orden: Desplazar → Invertir → Escalar amplitud')

---
## 🧠 Test Final — Autoevaluación

Responde las siguientes preguntas. Las celdas de código revelarán la respuesta correcta.

In [ ]:
# ================================================================
# TEST 1: ¿Qué sucede con x[0.5n]?
# ================================================================
# MODIFICA esta variable con tu respuesta antes de ejecutar:
# 'A' = La señal se comprime
# 'B' = La señal se expande y aparecen huecos (interpolación)
# 'C' = No cambia nada

mi_respuesta_1 = 'B'   # <- Escribe tu respuesta aquí

# ---------------------------------------------------------------
respuesta_correcta_1 = 'B'
if mi_respuesta_1 == respuesta_correcta_1:
    print('✅ ¡CORRECTO! x[0.5n] = x[n/2] EXPANDE la señal.')
    print('   Como n debe ser entero, a=0.5 genera índices fraccionarios.')
    print('   → Se necesita INTERPOLACIÓN para los valores faltantes.')
else:
    print('❌ Incorrecto. Pista: a=0.5 < 1 → expansión, no compresión.')
    print('   La compresión ocurre cuando a > 1 (ej. x[2n]).')

# Visualización de la respuesta
n = np.arange(-4, 9)
x_orig_v = np.sin(0.8 * n) * pulso_triangular(n.astype(float), centro=2, ancho=6)

n_exp = np.arange(-8, 17)  # expandido
x_exp = np.zeros(len(n_exp))
for i, ni in enumerate(n_exp):
    if ni % 2 == 0:
        orig_idx = np.where(n == ni // 2)[0]
        if len(orig_idx): x_exp[i] = x_orig_v[orig_idx[0]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Test 1: Efecto de $x[0.5n]$', fontsize=13, color='#00ffcc', fontweight='bold')

ml, sl, bl = ax1.stem(n, x_orig_v, basefmt='gray')
plt.setp(sl, color=COLOR_ORIG); plt.setp(ml, color=COLOR_ORIG)
ax1.set_title('Original $x[n]$'); ax1.grid(True)

mask_h = (n_exp % 2 != 0)
ml2, sl2, bl2 = ax2.stem(n_exp, x_exp, basefmt='gray')
plt.setp(sl2, color=COLOR_EXTRA); plt.setp(ml2, color=COLOR_EXTRA)
ax2.scatter(n_exp[mask_h], np.zeros(mask_h.sum()), marker='^',
            color='gray', s=60, zorder=5, alpha=0.7, label='Huecos (interpolación)')
ax2.set_title('$x[0.5n]$ — expandida con huecos'); ax2.grid(True); ax2.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ================================================================
# TEST 2: Orden correcto para z[n] = 0.5 * x[-n + 3]
# ================================================================
# Opciones:
# 'A' = Invertir → Desplazar 3 → Amplitud 0.5
# 'B' = Desplazar 3 a la izquierda → Invertir en n=0 → Amplitud 0.5
# 'C' = Amplitud 0.5 → Invertir → Desplazar

mi_respuesta_2 = 'B'   # <- Escribe tu respuesta aquí

respuesta_correcta_2 = 'B'
if mi_respuesta_2 == respuesta_correcta_2:
    print('✅ ¡CORRECTO!')
    print('   Factorizando: 0.5*x[-n+3] = 0.5*x[-(n-3)]')
    print('   Orden: 1) Desplazar +3 → 2) Invertir → 3) Amplitud ×0.5')
else:
    print('❌ Incorrecto. Recuerda la regla de oro:')
    print('   Primero factoriza: x[-n+3] = x[-(n-3)]')
    print('   Eso revela el desplazamiento +3 ANTES de la inversión.')

# Demostración visual del orden
def x_demo(n): return pulso_triangular(n.astype(float), centro=1, ancho=4)

n_base = np.arange(-5, 10)

paso_a = x_demo(n_base)            # x[n] original
paso_b = x_demo(n_base - 3)        # x[n-3]: desplazar
paso_c = x_demo(-(n_base - 3))     # x[-(n-3)]: invertir
paso_d = 0.5 * x_demo(-(n_base-3)) # 0.5 * x[-(n-3)]: amplitud

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Test 2: Orden correcto para $z[n] = 0.5x[-n+3]$',
             fontsize=12, color='#00ffcc', fontweight='bold')

datos_t2 = [
    (paso_a, COLOR_ORIG,  '①$x[n]$'),
    (paso_b, COLOR_EXTRA, '②$x[n-3]$ (desplazar +3)'),
    (paso_c, COLOR_TRANS, '③$x[-(n-3)]$ (invertir)'),
    (paso_d, COLOR_WARN,  '④$0.5x[-(n-3)]$ (amplitud) ✅'),
]

for ax, (y, col, titulo) in zip(axes, datos_t2):
    ml, sl, bl = ax.stem(n_base, y, basefmt='gray')
    plt.setp(sl, color=col); plt.setp(ml, color=col, markersize=7)
    ax.set_title(titulo, fontsize=9); ax.grid(True)
    ax.set_xlabel('$n$'); ax.set_ylim(-0.2, 1.3)
    ax.axvline(0, color='white', lw=0.5, ls=':')

plt.tight_layout()
plt.show()

---
## 📌 Resumen Final

| Concepto | Fórmula | Nota clave |
|----------|---------|------------|
| Desplazamiento | $x(t - t_0)$, $x[n-n_0]$ | $t_0 > 0$ retardo, $t_0 < 0$ adelanto |
| Escalamiento TC | $x(at)$ | $a > 1$ compresión, $0 < a < 1$ expansión |
| Escalamiento TD | $x[an]$ | Diezmado ($a > 1$) o interpolación ($a < 1$) |
| Inversión | $x(-t)$, $x[-n]$ | Espejo en eje vertical |
| Suma | $x_1 + x_2$ | Punto a punto |
| Producto | $x_1 \cdot x_2$ | Soporte = intersección |

### 🔑 Regla de Oro (orden de operaciones)
$$\underbrace{1.\;\text{Desplazamiento}}_{t_0\text{ o }n_0} \;\rightarrow\; \underbrace{2.\;\text{Inversión/Escalamiento}}_{a} \;\rightarrow\; \underbrace{3.\;\text{Amplitud}}_{A}$$

### ⚠️ Alerta TD
En tiempo discreto, **los índices SIEMPRE deben ser enteros**. Un desplazamiento no entero o un escalamiento que produzca índices fraccionarios **no tiene solución directa** sin algoritmos adicionales (interpolación).

---
*Módulo 7 — Operaciones con Señales | Procesamiento de Señales*